In [1]:
# 1

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.preprocessing import LabelEncoder

import xgboost as xgb
import networkx as nx
import ast


In [2]:
# 2

phy10s = pd.read_csv(
    r"C:\Users\amira\CISProj\CIS\exports\data\phys_agg_10s.csv",
    parse_dates=["bucket"],
)
phy16s = pd.read_csv(
    r"C:\Users\amira\CISProj\CIS\exports\data\phys_agg_16s.csv",
    parse_dates=["bucket"],
)
phy30s = pd.read_csv(
    r"C:\Users\amira\CISProj\CIS\exports\data\phys_agg_30s.csv",
    parse_dates=["bucket"],
)

phy10s["agg_window"] = 10
phy16s["agg_window"] = 16
phy30s["agg_window"] = 30

phyDf = pd.concat([phy10s, phy16s, phy30s], ignore_index=True)

print("Physical columns per file:")
print(set(phy10s.columns))
print(set(phy16s.columns))
print(set(phy30s.columns))

print("Physical duplicate buckets:", phyDf["bucket"].duplicated().sum())
display(phyDf.head())


Physical columns per file:
{'num_measurements', 'avg_value', 'agg_window', 'system_id', 'min_value', 'num_attacks', 'max_value', 'prop_key', 'asset_id', 'bucket', 'attack_types'}
{'num_measurements', 'avg_value', 'agg_window', 'system_id', 'min_value', 'num_attacks', 'max_value', 'prop_key', 'asset_id', 'bucket', 'attack_types'}
{'num_measurements', 'avg_value', 'agg_window', 'system_id', 'min_value', 'num_attacks', 'max_value', 'prop_key', 'asset_id', 'bucket', 'attack_types'}
Physical duplicate buckets: 84468


,bucket,system_id,prop_key,asset_id,avg_value,max_value,min_value,num_measurements,num_attacks,attack_types,agg_window
0,2021-04-09 11:30:50+00:00,testbed_system_1,pressure,testbed_system_1_Tank_Tank_1,31.8,178.0,0.0,10,0,['normal'],10
1,2021-04-09 11:30:50+00:00,testbed_system_1,pressure,testbed_system_1_Tank_Tank_2,0.0,0.0,0.0,10,0,['normal'],10
2,2021-04-09 11:30:50+00:00,testbed_system_1,pressure,testbed_system_1_Tank_Tank_3,0.0,0.0,0.0,10,0,['normal'],10
3,2021-04-09 11:30:50+00:00,testbed_system_1,pressure,testbed_system_1_Tank_Tank_4,0.0,0.0,0.0,10,0,['normal'],10
4,2021-04-09 11:30:50+00:00,testbed_system_1,pressure,testbed_system_1_Tank_Tank_5,0.0,0.0,0.0,10,0,['normal'],10


In [3]:
# 3

network10s = pd.read_csv(
    r"C:\Users\amira\CISProj\CIS\exports\data\scada_resolved_agg_10s.csv",
    parse_dates=["bucket"],
)
network16s = pd.read_csv(
    r"C:\Users\amira\CISProj\CIS\exports\data\scada_resolved_agg_16s.csv",
    parse_dates=["bucket"],
)
network30s = pd.read_csv(
    r"C:\Users\amira\CISProj\CIS\exports\data\scada_resolved_agg_30s.csv",
    parse_dates=["bucket"],
)

network10s["agg_window"] = 10
network16s["agg_window"] = 16
network30s["agg_window"] = 30

networkDf = pd.concat([network10s, network16s, network30s], ignore_index=True)

print("\nNetwork columns per file:")
print(set(network10s.columns))
print(set(network16s.columns))
print(set(network30s.columns))

print("Network duplicate buckets:", networkDf["bucket"].duplicated().sum())
display(networkDf.head())



Network columns per file:
{'destination_asset', 'agg_window', 'source_ip', 'destination_total_packets', 'system_id', 'avg_size', 'num_attacks', 'min_size', 'destination_mac', 'bucket', 'source_asset', 'source_total_packets', 'destination_ip', 'max_size', 'destination_key', 'protocol', 'source_mac', 'num_connections', 'attack_types', 'destination_port', 'source_port', 'source_key'}
{'destination_asset', 'agg_window', 'source_ip', 'destination_total_packets', 'system_id', 'avg_size', 'num_attacks', 'min_size', 'destination_mac', 'bucket', 'source_asset', 'source_total_packets', 'destination_ip', 'max_size', 'destination_key', 'protocol', 'source_mac', 'num_connections', 'attack_types', 'destination_port', 'source_port', 'source_key'}
{'destination_asset', 'agg_window', 'source_ip', 'destination_total_packets', 'system_id', 'avg_size', 'num_attacks', 'min_size', 'destination_mac', 'bucket', 'source_asset', 'source_total_packets', 'destination_ip', 'max_size', 'destination_key', 'protocol

,bucket,system_id,protocol,avg_size,source_total_packets,destination_total_packets,min_size,max_size,num_connections,source_ip,...,destination_ip,destination_port,destination_mac,source_key,destination_key,num_attacks,attack_types,source_asset,destination_asset,agg_window
0,2021-04-09 11:30:50+00:00,testbed_system_1,TCP,60.0,18,302,60,60,6,84.3.251.18,...,84.3.251.20,61514.0,74:46:a0:bd:a7:1b,84.3.251.18,84.3.251.20,0,['normal'],testbed_system_1_PLC_PLC_1,testbed_system_1_HMI_HMI_1,10
1,2021-04-09 11:30:50+00:00,testbed_system_1,Modbus,65.0,7,16,65,65,1,84.3.251.18,...,84.3.251.101,33829.0,e6:3f:ac:c9:a8:8c,84.3.251.18,84.3.251.101,0,['normal'],testbed_system_1_PLC_PLC_1,testbed_system_1_PLC_PLC_2,10
2,2021-04-09 11:30:50+00:00,testbed_system_1,TCP,60.0,44,104,60,60,6,84.3.251.18,...,84.3.251.101,33829.0,e6:3f:ac:c9:a8:8c,84.3.251.18,84.3.251.101,0,['normal'],testbed_system_1_PLC_PLC_1,testbed_system_1_PLC_PLC_2,10
3,2021-04-09 11:30:50+00:00,testbed_system_1,Modbus,65.0,7,16,65,65,1,84.3.251.18,...,84.3.251.101,34677.0,e6:3f:ac:c9:a8:8c,84.3.251.18,84.3.251.101,0,['normal'],testbed_system_1_PLC_PLC_1,testbed_system_1_PLC_PLC_2,10
4,2021-04-09 11:30:50+00:00,testbed_system_1,TCP,60.0,44,104,60,60,6,84.3.251.18,...,84.3.251.101,34677.0,e6:3f:ac:c9:a8:8c,84.3.251.18,84.3.251.101,0,['normal'],testbed_system_1_PLC_PLC_1,testbed_system_1_PLC_PLC_2,10


In [4]:
# 4

phys_numeric_cols = ["min_value", "max_value", "avg_value", "num_measurements"]

phyAgg = (
    phyDf
    .groupby("bucket")[phys_numeric_cols]
    .mean()
    .reset_index()
)

print("phyAgg shape:", phyAgg.shape)
display(phyAgg.head())


phyAgg shape: (1652, 5)


,bucket,min_value,max_value,avg_value,num_measurements
0,2021-04-09 11:30:30+00:00,0.000,4.5,0.815000,10.0
1,2021-04-09 11:30:40+00:00,0.000,0.0,0.000000,6.0
2,2021-04-09 11:30:50+00:00,0.000,4.5,0.815000,10.0
3,2021-04-09 11:30:56+00:00,0.075,23.1,11.242188,16.0
4,2021-04-09 11:31:00+00:00,5.800,33.9,20.069583,20.0


In [5]:
# 5

networkDf["bucket"] = pd.to_datetime(networkDf["bucket"])


networkDf["mac_src"] = networkDf["source_mac"].astype(str)
networkDf["mac_dst"] = networkDf["destination_mac"].astype(str)


networkDf["macport_src"] = (
    networkDf["source_mac"].astype(str) + ":" + networkDf["source_port"].astype(str)
)
networkDf["macport_dst"] = (
    networkDf["destination_mac"].astype(str) + ":" + networkDf["destination_port"].astype(str)
)


network_sorted = networkDf.sort_values("bucket").reset_index(drop=True)
time_values = network_sorted["bucket"].to_numpy()

print("network_sorted shape:", network_sorted.shape)
display(network_sorted.head())


network_sorted shape: (1058136, 26)


,bucket,system_id,protocol,avg_size,source_total_packets,destination_total_packets,min_size,max_size,num_connections,source_ip,...,destination_key,num_attacks,attack_types,source_asset,destination_asset,agg_window,mac_src,mac_dst,macport_src,macport_dst
0,2021-04-09 11:30:30+00:00,testbed_system_1,TCP,64.666667,88,38,60,74,6,84.3.251.101,...,84.3.251.18,0,['normal'],testbed_system_1_PLC_PLC_2,testbed_system_1_PLC_PLC_1,30,e6:3f:ac:c9:a8:8c,00:80:f4:03:fb:12,e6:3f:ac:c9:a8:8c:51329.0,00:80:f4:03:fb:12:502.0
1,2021-04-09 11:30:30+00:00,testbed_system_1,TCP,71.600000,69,10,66,78,5,84.3.251.102,...,84.3.251.104,0,['normal'],testbed_system_1_PLC_PLC_3,testbed_system_1_Flow Sensor_Flow_sensor_1,30,0a:fe:ec:47:74:fb,4a:35:83:e0:3d:a4,0a:fe:ec:47:74:fb:502.0,4a:35:83:e0:3d:a4:53315.0
2,2021-04-09 11:30:30+00:00,testbed_system_1,TCP,71.600000,73,10,66,78,5,84.3.251.102,...,84.3.251.104,0,['normal'],testbed_system_1_PLC_PLC_3,testbed_system_1_Flow Sensor_Flow_sensor_1,30,0a:fe:ec:47:74:fb,4a:35:83:e0:3d:a4,0a:fe:ec:47:74:fb:502.0,4a:35:83:e0:3d:a4:58691.0
3,2021-04-09 11:30:30+00:00,testbed_system_1,Modbus,78.000000,17,18,78,78,1,84.3.251.102,...,84.3.251.101,0,['normal'],testbed_system_1_PLC_PLC_3,testbed_system_1_PLC_PLC_2,30,0a:fe:ec:47:74:fb,e6:3f:ac:c9:a8:8c,0a:fe:ec:47:74:fb:50105.0,e6:3f:ac:c9:a8:8c:502.0
4,2021-04-09 11:30:30+00:00,testbed_system_1,TCP,68.666667,102,104,66,74,6,84.3.251.102,...,84.3.251.101,0,['normal'],testbed_system_1_PLC_PLC_3,testbed_system_1_PLC_PLC_2,30,0a:fe:ec:47:74:fb,e6:3f:ac:c9:a8:8c,0a:fe:ec:47:74:fb:50105.0,e6:3f:ac:c9:a8:8c:502.0


In [6]:
# 6

def compute_graph_metrics_for_slice(df_slice, src_col, dst_col):
    """
    Build directed graph from df_slice[src_col] -> df_slice[dst_col]
    and compute: num_nodes, num_edges, avg_degree, density.
    """

    if df_slice.empty:
        return 0, 0, 0.0, 0.0

    G = nx.DiGraph()
    edges = list(zip(df_slice[src_col], df_slice[dst_col]))
    G.add_edges_from(edges)

    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()

    if num_nodes == 0:
        avg_degree = 0.0
    else:
        degrees = dict(G.degree())
        avg_degree = float(sum(degrees.values())) / num_nodes

    density = nx.density(G)

    return num_nodes, num_edges, avg_degree, density


In [7]:
# 7

unique_times = np.unique(time_values)

window_1m = pd.Timedelta(seconds=60)
window_5m = pd.Timedelta(seconds=300)

records = []

for t in unique_times:
   
    start_1m = time_values.searchsorted(t - window_1m, side="left")
    end_1m   = time_values.searchsorted(t,            side="right")
    slice_1m = network_sorted.iloc[start_1m:end_1m]

  
    start_5m = time_values.searchsorted(t - window_5m, side="left")
    end_5m   = end_1m
    slice_5m = network_sorted.iloc[start_5m:end_5m]

   
    mac_nodes_1m, mac_edges_1m, mac_deg_1m, mac_density_1m = compute_graph_metrics_for_slice(
        slice_1m, "mac_src", "mac_dst"
    )
    mac_nodes_5m, mac_edges_5m, mac_deg_5m, mac_density_5m = compute_graph_metrics_for_slice(
        slice_5m, "mac_src", "mac_dst"
    )


    mp_nodes_1m, mp_edges_1m, mp_deg_1m, mp_density_1m = compute_graph_metrics_for_slice(
        slice_1m, "macport_src", "macport_dst"
    )
    mp_nodes_5m, mp_edges_5m, mp_deg_5m, mp_density_5m = compute_graph_metrics_for_slice(
        slice_5m, "macport_src", "macport_dst"
    )

    records.append({
        "bucket": t,

  
        "mac_nodes_1m": mac_nodes_1m,
        "mac_edges_1m": mac_edges_1m,
        "mac_avg_degree_1m": mac_deg_1m,
        "mac_density_1m": mac_density_1m,

        "mac_nodes_5m": mac_nodes_5m,
        "mac_edges_5m": mac_edges_5m,
        "mac_avg_degree_5m": mac_deg_5m,
        "mac_density_5m": mac_density_5m,


        "mp_nodes_1m": mp_nodes_1m,
        "mp_edges_1m": mp_edges_1m,
        "mp_avg_degree_1m": mp_deg_1m,
        "mp_density_1m": mp_density_1m,

        "mp_nodes_5m": mp_nodes_5m,
        "mp_edges_5m": mp_edges_5m,
        "mp_avg_degree_5m": mp_deg_5m,
        "mp_density_5m": mp_density_5m,
    })

graph_metrics_df = pd.DataFrame.from_records(records)

print("Graph metrics shape:", graph_metrics_df.shape)
display(graph_metrics_df.head())


Graph metrics shape: (1649, 17)


,bucket,mac_nodes_1m,mac_edges_1m,mac_avg_degree_1m,mac_density_1m,mac_nodes_5m,mac_edges_5m,mac_avg_degree_5m,mac_density_5m,mp_nodes_1m,mp_edges_1m,mp_avg_degree_1m,mp_density_1m,mp_nodes_5m,mp_edges_5m,mp_avg_degree_5m,mp_density_5m
0,2021-04-09 11:30:30+00:00,7,18,5.142857,0.428571,7,18,5.142857,0.428571,50,92,3.680000,0.037551,50,92,3.680000,0.037551
1,2021-04-09 11:30:40+00:00,7,18,5.142857,0.428571,7,18,5.142857,0.428571,50,92,3.680000,0.037551,50,92,3.680000,0.037551
2,2021-04-09 11:30:50+00:00,7,18,5.142857,0.428571,7,18,5.142857,0.428571,50,92,3.680000,0.037551,50,92,3.680000,0.037551
3,2021-04-09 11:30:56+00:00,7,18,5.142857,0.428571,7,18,5.142857,0.428571,123,238,3.869919,0.015860,123,238,3.869919,0.015860
4,2021-04-09 11:31:00+00:00,7,18,5.142857,0.428571,7,18,5.142857,0.428571,233,458,3.931330,0.008473,233,458,3.931330,0.008473


In [8]:
# 8

graph_metrics_df["bucket"] = pd.to_datetime(graph_metrics_df["bucket"])
phyAgg["bucket"]          = pd.to_datetime(phyAgg["bucket"])
networkDf["bucket"]       = pd.to_datetime(networkDf["bucket"])


combinedDf = pd.merge_asof(
    networkDf.sort_values("bucket"),
    phyAgg.sort_values("bucket"),
    on="bucket",
    direction="backward",
)


combinedDf = combinedDf.merge(graph_metrics_df, on="bucket", how="left")


graph_cols = [c for c in graph_metrics_df.columns if c != "bucket"]
combinedDf[graph_cols] = combinedDf[graph_cols].fillna(0)

print("combinedDf shape:", combinedDf.shape)
display(combinedDf.head())


combinedDf shape: (1058136, 46)


,bucket,system_id,protocol,avg_size,source_total_packets,destination_total_packets,min_size,max_size,num_connections,source_ip,...,mac_avg_degree_5m,mac_density_5m,mp_nodes_1m,mp_edges_1m,mp_avg_degree_1m,mp_density_1m,mp_nodes_5m,mp_edges_5m,mp_avg_degree_5m,mp_density_5m
0,2021-04-09 11:30:30+00:00,testbed_system_1,TCP,64.666667,88,38,60,74,6,84.3.251.101,...,5.142857,0.428571,50,92,3.68,0.037551,50,92,3.68,0.037551
1,2021-04-09 11:30:30+00:00,testbed_system_1,TCP,71.600000,69,10,66,78,5,84.3.251.102,...,5.142857,0.428571,50,92,3.68,0.037551,50,92,3.68,0.037551
2,2021-04-09 11:30:30+00:00,testbed_system_1,TCP,71.600000,73,10,66,78,5,84.3.251.102,...,5.142857,0.428571,50,92,3.68,0.037551,50,92,3.68,0.037551
3,2021-04-09 11:30:30+00:00,testbed_system_1,Modbus,78.000000,17,18,78,78,1,84.3.251.102,...,5.142857,0.428571,50,92,3.68,0.037551,50,92,3.68,0.037551
4,2021-04-09 11:30:30+00:00,testbed_system_1,TCP,68.666667,102,104,66,74,6,84.3.251.102,...,5.142857,0.428571,50,92,3.68,0.037551,50,92,3.68,0.037551


In [9]:
# 9

def is_attack(s):
    if pd.isna(s):
        return 0 
    try:
        labels = ast.literal_eval(s)
        labels = [lbl.lower() for lbl in labels]
    except:
        labels = [str(s).lower()]


    if "normal" in labels:
        return 0


    return 1

combinedDf["binary_attack"] = combinedDf["attack_types"].apply(is_attack)

print("Binary attack label distribution (0=Normal, 1=Attack):")
print(combinedDf["binary_attack"].value_counts())


Binary attack label distribution (0=Normal, 1=Attack):
binary_attack
1    564817
0    493319
Name: count, dtype: int64


In [10]:
# 10

y = combinedDf["binary_attack"]

drop_cols = [
    "bucket",
    "attack_types",
    "binary_attack",
    "attack_class",      
    "num_attacks",


    "label",
    "label_n",


    "modbus_fn",
    "modbus_response",
    "n_pkt_src",
    "n_pkt_dst",


    "source_ip", "destination_ip",
    "source_mac", "destination_mac",
    "source_key", "destination_key",
    "system_id",
]

X = combinedDf.drop(columns=[c for c in drop_cols if c in combinedDf.columns])

print("Initial X shape:", X.shape)
print("Some feature columns:", X.columns[:20].tolist())


Initial X shape: (1058136, 36)
Some feature columns: ['protocol', 'avg_size', 'source_total_packets', 'destination_total_packets', 'min_size', 'max_size', 'num_connections', 'source_port', 'destination_port', 'source_asset', 'destination_asset', 'agg_window', 'mac_src', 'mac_dst', 'macport_src', 'macport_dst', 'min_value', 'max_value', 'avg_value', 'num_measurements']


In [11]:
# 11

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
label_encoders = {}

print("Categorical feature columns:", cat_cols)

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

print("Final X shape after encoding:", X.shape)


Categorical feature columns: ['protocol', 'source_asset', 'destination_asset', 'mac_src', 'mac_dst', 'macport_src', 'macport_dst']
Final X shape after encoding: (1058136, 36)


In [12]:
# 12

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Train label distribution:", np.bincount(y_train))
print("Test label distribution :", np.bincount(y_test))


Train shape: (846508, 36)  Test shape: (211628, 36)
Train label distribution: [394655 451853]
Test label distribution : [ 98664 112964]


In [13]:
# 13

xgb_binary = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    n_jobs=-1,
)

xgb_binary.fit(X_train, y_train)

y_pred_prob = xgb_binary.predict_proba(X_test)[:, 1]
y_pred = (y_pred_prob > 0.5).astype(int)


In [16]:
# 14

acc = accuracy_score(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)

print(f"Binary anomaly detection Accuracy: {acc:.4f}")
print(f"Binary anomaly detection Balanced Accuracy: {bal_acc:.4f}\n")

print("report (0=Normal, 1=Attack):")
print(classification_report(y_test, y_pred, target_names=["Normal", "Attack"]))

labels = [0, 1]
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

print("\n TPR / FPR:")
for lbl, row in zip(labels, cm_norm):
    tpr = row[labels.index(lbl)]
    fpr = 1 - tpr
    name = "Normal" if lbl == 0 else "Attack"
    print(f"{name:7s} (label={lbl})  TPR={tpr:.4f}  FPR={fpr:.4f}")


Binary anomaly detection Accuracy: 0.9898
Binary anomaly detection Balanced Accuracy: 0.9899

report (0=Normal, 1=Attack):
              precision    recall  f1-score   support

      Normal       0.99      0.99      0.99     98664
      Attack       0.99      0.99      0.99    112964

    accuracy                           0.99    211628
   macro avg       0.99      0.99      0.99    211628
weighted avg       0.99      0.99      0.99    211628


 TPR / FPR:
Normal  (label=0)  TPR=0.9905  FPR=0.0095
Attack  (label=1)  TPR=0.9892  FPR=0.0108
